# Travel Assistant

## Возможности
- RAG (поиск по политике компании из `data/policies.txt`)
- Поиск авиабилетов из CSV (`data/tickets.csv`)
- Анализ бюджета (лимиты из политики)
- Подбор отеля (с учетом бюджета и Wi-Fi)
- **Генерация финального ответа через LLM** (OpenRouter или локальная модель)
- Управление через `.env`

## Конфигурация
Создайте файл `.env` в той же директории (см. пример ниже) или используйте переменные окружения в Colab.

```bash
LLM_PROVIDER=openrouter
OPENROUTER_API_KEY=sk-or-v1-...
OPENROUTER_MODEL=deepseek/deepseek-r1:free
# или
# LLM_PROVIDER=local
# LOCAL_MODEL_NAME=llama3.2
# LOCAL_BASE_URL=http://localhost:11434/v1
```

## 1. Установка зависимостей

In [12]:
!pip install -q langchain langgraph langchain-community chromadb sentence-transformers torch pandas openai python-dotenv langchain-openai

## 2. Конфигурация (загрузка .env и инициализация LLM)

In [2]:
import os
from typing import Dict, List, Any
import pandas as pd
from io import StringIO
import traceback
from dotenv import load_dotenv

# Загружаем .env (если файл есть)
load_dotenv()

# === Чтение конфигурации ===
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openrouter").lower()

# OpenRouter
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-r1:free")

# Локальная (Ollama / LM Studio)
LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "llama3.2")
LOCAL_BASE_URL = os.getenv("LOCAL_BASE_URL", "http://localhost:11434")
LOCAL_API_KEY = os.getenv("LOCAL_API_KEY", "ollama")

# Эмбеддинги
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

# Логирование
LOG_LEVEL = os.getenv("LOG_LEVEL", "info").upper()

# === Инициализация LLM ===
llm = None

if LLM_PROVIDER == "openrouter" and OPENROUTER_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        model=OPENROUTER_MODEL,
        temperature=0.7
    )
    print(f"✅ LLM инициализирован: OpenRouter / {OPENROUTER_MODEL}")
    # Быстрый пинг
    try:
        response = llm.invoke("Ответь OK")
        print(f"✅ Пинг успешен: {response.content[:50]}")
    except Exception as e:
        print(f"❌ Пинг провален: {e}")
        
elif LLM_PROVIDER == "local":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=LOCAL_API_KEY,
        base_url=LOCAL_BASE_URL,
        model=LOCAL_MODEL_NAME,
        temperature=0.7
    )
    print(f"✅ LLM инициализирован: локальная модель {LOCAL_MODEL_NAME} ({LOCAL_BASE_URL})")
    # Быстрый пинг
    try:
        response = llm.invoke("Ответь OK")
        print(f"✅ Пинг успешен: {response.content[:50]}")
    except Exception as e:
        print(f"❌ Пинг провален: {e}")
        
else:
    print("⚠️ LLM не сконфигурирован (нет API ключа или выбран неподдерживаемый провайдер).")
    print("   Будет использована заглушка DummyLLM, которая не генерирует осмысленные ответы.")
    class DummyLLM:
        def invoke(self, prompt, **kwargs):
            return f"[DummyLLM] Нет реального LLM. Промпт: {prompt[:200]}..."
    llm = DummyLLM()

def print_config():
    print("\n" + "="*50)
    print("ТЕКУЩАЯ КОНФИГУРАЦИЯ")
    print("="*50)
    print(f"LLM Provider: {LLM_PROVIDER.upper()}")
    if LLM_PROVIDER == "openrouter":
        print(f"Model: {OPENROUTER_MODEL}")
        print(f"API Key: {'***' if OPENROUTER_API_KEY else 'НЕТ'}")
    elif LLM_PROVIDER == "local":
        print(f"Model: {LOCAL_MODEL_NAME}")
        print(f"Base URL: {LOCAL_BASE_URL}")
    print(f"Embedding: {EMBEDDING_MODEL}")
    print("="*50 + "\n")

print_config()

✅ LLM инициализирован: локальная модель qwen/qwen3.5-9b (http://127.0.0.1:1234/v1/)
✅ Пинг успешен: 

OK

ТЕКУЩАЯ КОНФИГУРАЦИЯ
LLM Provider: LOCAL
Model: qwen/qwen3.5-9b
Base URL: http://127.0.0.1:1234/v1/
Embedding: sentence-transformers/all-MiniLM-L6-v2



## 3. Загрузка данных из файлов

In [3]:
# Загрузка CSV с билетами из data/tickets.csv
tickets_df = pd.read_csv("data/tickets.csv")
print(f"✅ Загружено {len(tickets_df)} рейсов из data/tickets.csv")
print(tickets_df.head())

# Загрузка политики из data/policies.txt
with open("data/policies.txt", "r", encoding="utf-8") as f:
    policy_text = f.read()
print(f"✅ Загружен текст политики ({len(policy_text)} символов) из data/policies.txt")
print(policy_text[:500] + "..." if len(policy_text) > 500 else policy_text)

✅ Загружено 203 рейсов из data/tickets.csv
       airline flight_number departure_city     arrival_city departure_date  \
0     Аэрофлот        SU1001         Москва  Санкт-Петербург     2026-06-01   
1  S7 Airlines         S7002         Москва  Санкт-Петербург     2026-06-01   
2       Победа        DP1003         Москва  Санкт-Петербург     2026-06-01   
3        UTAir         U1004         Москва  Санкт-Петербург     2026-06-01   
4     Azur Air        ZA1005         Москва  Санкт-Петербург     2026-06-01   

   price  is_direct  
0  15200          1  
1  13400          1  
2   8900          0  
3  11800          1  
4  10700          1  
✅ Загружен текст политики (983 символов) из data/policies.txt
=== ПОЛИТИКА КОМАНДИРОВОК КОМПАНИИ ===

1. АВИАБИЛЕТЫ

- Эконом-класс для перелетов до 4 часов.
- Бизнес-класс разрешён при перелетах более 6 часов или для сотрудников уровня директор и выше.
- Максимальная стоимость билета не должна превышать 50 000 рублей в одну сторону.
- Рекомендуетс

## 4. RAG: эмбеддинги, чанкинг, векторное хранилище

In [4]:
import shutil
import os

if os.path.exists('.data/.chroma'):
    shutil.rmtree('.data/.chroma')
    print("🗑️ Старая база удалена")

🗑️ Старая база удалена


In [5]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Эмбеддер (с fallback)
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    print("✅ Эмбеддер загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки эмбеддера: {e}")
    class DummyEmbeddings:
        def embed_documents(self, texts): return [[0.0]*384 for _ in texts]
        def embed_query(self, text): return [0.0]*384
    embeddings = DummyEmbeddings()
    print("⚠️ Используется DummyEmbeddings")

# Документы
policy_doc = Document(page_content=policy_text, metadata={"source": "policy"})

ticket_docs = []
for _, row in tickets_df.iterrows():
    content = (
        f"Рейс {row['flight_number']} "
        f"{row['airline']} "
        f"{row['departure_city']} → {row['arrival_city']} "
        f"{row['departure_date']}, "
        f"{row['price']} руб., "
        f"{'прямой' if row['is_direct'] else 'с пересадкой'}"
    )
    metadata = {
        "source": "ticket",
        "flight_number": row["flight_number"],
        "departure_city": row["departure_city"],
        "arrival_city": row["arrival_city"],
        "price": row["price"],
        "departure_date": row["departure_date"]
    }
    ticket_docs.append(Document(page_content=content, metadata=metadata))

# Чанкинг только для политики
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
policy_chunks = splitter.split_documents([policy_doc])
print(f"Политика разбита на {len(policy_chunks)} чанков")

# Векторное хранилище (два коллектора: policy и tickets)
vectorstore = Chroma.from_documents(documents= policy_chunks + ticket_docs,
                                    embedding=embeddings,
                                    persist_directory='.data/.chroma' )
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Vector store готов")

C:\Users\semva\AppData\Local\Temp\ipykernel_9876\3310058664.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Эмбеддер загружен
Политика разбита на 3 чанков
✅ Vector store готов


## 5. Агенты и граф (с использованием LLM для финального ответа)

In [8]:
import re
from datetime import datetime
from typing import Tuple, Optional, List, Dict
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class ParsedTravelQuery(BaseModel):
    departure_city: Optional[str] = Field(None, description="Город вылета")
    arrival_city: Optional[str] = Field(None, description="Город прилёта")
    departure_date: Optional[str] = Field(
        None, description="Дата вылета в формате YYYY-MM-DD"
    )
    prefer_direct: bool = Field(False, description="Предпочтение прямых рейсов")



# ------------------- LLM парсер (медленный, но умный) -------------------
def llm_extract_query(query: str, llm) -> ParsedTravelQuery:
    """Вызывает LLM для извлечения параметров поездки."""
    parser = PydanticOutputParser(pydantic_object=ParsedTravelQuery)
    prompt = f"""
Ты – помощник по командировкам. Извлеки из запроса пользователя параметры поездки.

Запрос: {query}

{parser.get_format_instructions()}

Правила:
- Города пиши в именительном падеже (Москва, Санкт-Петербург).
- Дату приведи в формат YYYY-MM-DD. Если дата относительная ("завтра"), считай от сегодняшнего дня: {datetime.now().date()}.
- Если параметр не указан, оставь null.
"""
    try:
        response = llm.invoke(prompt)
        parsed = parser.parse(response.content)
        return parsed
    except Exception as e:
        print(f"   LLM парсинг не удался: {e}")
        return ParsedTravelQuery()  # пустая модель

In [9]:
from langgraph.graph import StateGraph, END
from typing import TypedDict


class AgentState(TypedDict):
    user_query: str
    policy_context: List[str]
    tickets: List[Dict]
    budget_ok: bool
    recommended_hotel: Dict
    final_answer: str
    parsed_query: Dict


class QueryParser:
    """Извлекает города, дату и предпочтения из запроса (гибрид: RegEx + LLM fallback)."""

    def __init__(self, llm):
        self.llm = llm

    def invoke(self, state: AgentState) -> AgentState:
        print("🧠 Агент QueryParser: извлечение параметров ")
        query = state["user_query"]

        # RegEx не смог → вызываем LLM
        parsed = self._llm_extract(query)

        state["parsed_query"] = parsed
        print(f"   Распознано: {parsed}")
        return state

    def _llm_extract(self, query: str) -> Dict:
        """Вызов LLM с Pydantic-парсером."""
        from langchain_core.output_parsers import PydanticOutputParser

        parser = PydanticOutputParser(pydantic_object=ParsedTravelQuery)
        prompt = f"""
Ты – помощник по командировкам. Извлеки из запроса параметры поездки.

Запрос: {query}

{parser.get_format_instructions()}

Правила:
- Города пиши в именительном падеже (Москва, Санкт-Петербург).
- Дату приведи в формат YYYY-MM-DD. Если дата относительная ("завтра"), считай от {datetime.now().date()}.
- prefer_direct = True, если пользователь сказал "прямой рейс" или "без пересадок".
"""
        try:
            resp = self.llm.invoke(prompt)
            parsed_obj = parser.parse(resp.content)
            return parsed_obj.model_dump()
        except Exception as e:
            print(f"   Ошибка LLM: {e}")
            return {}


# ---------- Агент 1: Поиск политики (RAG) ----------
class PolicyExpert:
    def invoke(self, state: AgentState) -> AgentState:
        print("📋 Агент PolicyExpert: поиск релевантных правил...")
        
        # Ищем ТОЛЬКО документы с source=policy
        policy_docs = vectorstore.similarity_search(
            state["user_query"], 
            k=3, 
            filter={"source": "policy"}
        )
        
        state["policy_context"] = [d.page_content for d in policy_docs]
        print(f"   Найдено {len(state['policy_context'])} фрагментов политики")
        
        # Fallback: если ничего не нашлось, берём общую выдачу и фильтруем
        if not state["policy_context"]:
            all_docs = retriever.invoke(state["user_query"])
            state["policy_context"] = [d.page_content for d in all_docs if d.metadata.get("source") == "policy"]
            
        return state


# ---------- Агент 2: Поиск билетов ----------
class TicketSearcher:
    def invoke(self, state: AgentState) -> AgentState:
        print("🔍 Агент TicketSearcher: поиск билетов по распарсенным параметрам...")
        parsed = state.get("parsed_query", {})
        from_city = parsed.get("departure_city")
        to_city = parsed.get("arrival_city")
        date_str = parsed.get("departure_date")

        if not from_city or not to_city:
            print("   ⚠️ Города не определены, поиск отменён.")
            state["tickets"] = []
            return state

        filtered = tickets_df.copy()
        filtered = filtered[
            (filtered["departure_city"].str.lower() == from_city.lower()) &
            (filtered["arrival_city"].str.lower() == to_city.lower())
        ]

        if date_str:
            filtered = filtered[filtered["departure_date"] == date_str]
            print(f"   Фильтр по дате: {date_str}")

        if parsed.get("prefer_direct"):
            filtered = filtered.sort_values(by=["is_direct", "price"], ascending=[False, True])

        state["tickets"] = filtered.to_dict(orient="records")
        print(f"   Найдено билетов: {len(state['tickets'])}")
        return state


# ---------- Агент 3: Бюджетный контроль ----------
class BudgetAnalyst:
    def invoke(self, state: AgentState) -> AgentState:
        print("💰 Агент BudgetAnalyst: проверка лимитов...")
        if not state["tickets"]:
            state["budget_ok"] = False
            return state
        # Простейший лимит из политики (можно вытащить из policy_context)
        MAX_PRICE = 50000
        state["budget_ok"] = any(t["price"] <= MAX_PRICE for t in state["tickets"])
        print(f"   Бюджет {'соблюдён' if state['budget_ok'] else 'превышен'}")
        return state


# ---------- Агент 4: Рекомендация отеля ----------
class HotelBooker:
    def invoke(self, state: AgentState) -> AgentState:
        print("🏨 Агент HotelBooker: подбор отеля...")
        if not state["tickets"]:
            state["recommended_hotel"] = {}
            return state
        # База отелей
        hotels_db = {
            "Санкт-Петербург": {
                "name": "Отель \u0022Амбассадор\u0022",
                "price": 7200,
                "wifi": True,
            },
            "Казань": {"name": "Ramada Kazan", "price": 5900, "wifi": True},
            "Новосибирск": {"name": "Marins Park", "price": 4300, "wifi": True},
            "Сочи": {"name": "Жемчужина", "price": 7800, "wifi": True},
            "Москва": {"name": "Аэростар", "price": 6500, "wifi": True},
        }
        arrival = state["tickets"][0]["arrival_city"]
        if arrival in hotels_db:
            state["recommended_hotel"] = hotels_db[arrival]
            print(
                f"   Рекомендован отель: {state['recommended_hotel']['name']} ({state['recommended_hotel']['price']} руб/ночь)"
            )
        else:
            state["recommended_hotel"] = {
                "name": "нет данных",
                "price": 0,
                "wifi": False,
            }
            print("   Отель не найден в базе")
        return state


# ---------- Агент 5: Генерация финального ответа через LLM ----------
class FinalAnswerGenerator:
    def __init__(self, llm):
        self.llm = llm

    def invoke(self, state: AgentState) -> AgentState:
        print("🤖 Агент FinalAnswerGenerator: формирование ответа с помощью LLM...")
        # Собираем контекст для LLM
        context = {
            "query": state["user_query"],
            "policy": (
                "\n".join(state["policy_context"])
                if state["policy_context"]
                else "(нет данных)"
            ),
            "tickets": state["tickets"],
            "budget_ok": state["budget_ok"],
            "hotel": state["recommended_hotel"],
        }

        prompt = f"""Ты — помощник по командировкам. Ответь пользователю на русском языке, вежливо и полезно.

Запрос пользователя: {context['query']}

Информация из политики компании:
{context['policy']}

Найденные билеты:
{chr(10).join([f"- {t['flight_number']}: {t['departure_city']} \u2192 {t['arrival_city']}, {t['price']} руб., {'прямой' if t['is_direct'] else 'с пересадкой'}" for t in context['tickets']]) if context['tickets'] else 'Билеты не найдены'}
Соответствие бюджету: {'Да' if context['budget_ok'] else 'Нет (превышает лимит 50 000 руб)'}
Рекомендуемый отель: {context['hotel'].get('name', 'нет')} ({context['hotel'].get('price', 0)} руб/ночь, Wi-Fi: {'есть' if context['hotel'].get('wifi') else 'нет'})
Сформируй понятный, дружелюбный ответ, который суммирует информацию и даёт рекомендации. Если билетов нет — предложи альтернативу. Если отель не найден — посоветуй поискать самостоятельно."""

        try:
            response = self.llm.invoke(prompt)
            state["final_answer"] = (
                response.content if hasattr(response, "content") else str(response)
            )
        except Exception as e:
            print(f"   Ошибка LLM: {e}")
            state["final_answer"] = (
                f"Извините, произошла ошибка при генерации ответа. Детали: {e}"
            )

        return state


# ---------- Сборка графа ----------
workflow = StateGraph(AgentState)

workflow.add_node("query_parser", QueryParser(llm).invoke)
workflow.add_node("policy_expert", PolicyExpert().invoke)
workflow.add_node("ticket_searcher", TicketSearcher().invoke)
workflow.add_node("budget_analyst", BudgetAnalyst().invoke)
workflow.add_node("hotel_booker", HotelBooker().invoke)
workflow.add_node("final_answer", FinalAnswerGenerator(llm).invoke)


workflow.set_entry_point("query_parser")
workflow.add_edge("query_parser", "policy_expert")
workflow.add_edge("policy_expert", "ticket_searcher")
workflow.add_edge("ticket_searcher", "budget_analyst")
workflow.add_edge("budget_analyst", "hotel_booker")
workflow.add_edge("hotel_booker", "final_answer")
workflow.add_edge("final_answer", END)

app = workflow.compile()
print("✅ Граф агентов готов")

✅ Граф агентов готов


## 6. Запуск и тесты

In [11]:
def run_query(query: str):
    print(f"\n{'='*60}\nЗАПРОС: {query}\n{'='*60}")
    initial_state = {
        "user_query": query,
        "parsed_query": {},      
        "policy_context": [],
        "tickets": [],
        "budget_ok": False,
        "recommended_hotel": {},
        "final_answer": ""
    }
    try:
        result = app.invoke(initial_state)
        print("\n✅ ОТВЕТ АССИСТЕНТА:\n")
        print(result["final_answer"])
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        traceback.print_exc()

# Тест 1: Поиск билетов в Питер
run_query("Нужны билеты из Москвы в Санкт-Петербург с вылетом 01.06.2026")

# Тест 2: Вопрос о правилах
run_query("Какие ограничения по стоимости авиабилетов в командировках?")

# Тест 3: Билет в город без отеля
run_query("Нужен билет из Екатеринбурга в Москву")


ЗАПРОС: Нужны билеты из Москвы в Санкт-Петербург с вылетом 01.06.2026
🧠 Агент QueryParser: извлечение параметров 
   Распознано: {'departure_city': 'Москва', 'arrival_city': 'Санкт-Петербург', 'departure_date': '2026-06-01', 'prefer_direct': False}
📋 Агент PolicyExpert: поиск релевантных правил...
   Найдено 3 фрагментов политики
🔍 Агент TicketSearcher: поиск билетов по распарсенным параметрам...
   Фильтр по дате: 2026-06-01
   Найдено билетов: 5
💰 Агент BudgetAnalyst: проверка лимитов...
   Бюджет соблюдён
🏨 Агент HotelBooker: подбор отеля...
   Рекомендован отель: Отель "Амбассадор" (7200 руб/ночь)
🤖 Агент FinalAnswerGenerator: формирование ответа с помощью LLM...

✅ ОТВЕТ АССИСТЕНТА:



Здравствуйте! Рад помочь вам с организацией командировки.

Я подготовил подборку билетов из Москвы в Санкт-Петербург на **1 июня 2026 года**, которые соответствуют внутренней политике компании. Все найденные варианты укладываются в бюджет (максимум 50 000 руб.) и позволяют оформить расходы как служ

## 7. Файл .env (пример)

Сохраните этот файл как `.env` в той же папке, что и ноутбук:

```ini
# ==========================================
# Конфигурация Travel Assistant
# ==========================================

# Выбор провайдера LLM: 'openrouter' или 'local'
LLM_PROVIDER=openrouter

# --- OpenRouter настройки ---
OPENROUTER_API_KEY=ваш_ключ_здесь
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=deepseek/deepseek-r1:free

# --- Локальные модели (Ollama / LM Studio) ---
LOCAL_MODEL_NAME=llama3.2
LOCAL_BASE_URL=http://localhost:11434/v1
LOCAL_API_KEY=ollama

# --- Эмбеддинги ---
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2

# --- Общие ---
LOG_LEVEL=info
```

## Готово!

Теперь `.env` действительно управляет выбором LLM. Ответы генерируются нейросетью с учётом политики из `data/policies.txt`, билетов из `data/tickets.csv`, бюджета и отелей. Удачных командировок! 🚀